# Demo MDLM

In [ ]:
import os, sys, math, time, json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10
from torchvision.transforms import v2
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

print(f"device   : {device}", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print(f"cpu count: {os.cpu_count()}")

In [3]:
print(logits.exp().sum(-1)[0, :3])      # ≈ 1 → ce sont déjà des log-probas
print(logits[0, 5, 50257])               # -inf → masque interdit, SUBS appliqué
print(logits[0, 0].argmax(), x[0, 0])    # égaux → carry-over appliqué

tensor([1.1202e+10, 3.3627e+10, 4.6905e+12], device='cuda:0')
tensor(0., device='cuda:0')
tensor(383, device='cuda:0') tensor(464, device='cuda:0')


In [ ]:
@torch.no_grad()
def mdlm_step(model, x, t, s, mask_id):
    B, L = x.shape
    sigma = torch.zeros(B, device=x.device)
    logits = model(x, sigma)
    logits = logits[..., :mask_id]
    probs = logits.softmax(-1)
    V = probs.shape[-1]
    token = torch.multinomial(probs.view(B * L, V), num_samples=1).view(B, L)

    u = torch.rand(x.shape, device=x.device)
    est_masque = (x == mask_id)
    reste_masque = u < s / t

    interieur = torch.where(reste_masque, mask_id, token)
    x_next = torch.where(est_masque, interieur, x)
    return x_next


def mdlm_sample(model, B, L, T, mask_id):
    x = torch.full((B, L), mask_id, dtype=torch.long, device=next(model.parameters()).device)
    grid = torch.linspace(1, 0, T+1)
    for i in range(T):
        x_next = mdlm_step(model, x, grid[i], grid[i+1], mask_id)
        assert (x_next == mask_id).sum() <= (x == mask_id).sum()
        deja_revele = (x != mask_id)
        assert torch.equal(x_next[deja_revele], x[deja_revele])
        x = x_next
    return x

In [ ]:
import torch, transformers

tok = transformers.AutoTokenizer.from_pretrained("gpt2")
model = transformers.AutoModelForMaskedLM.from_pretrained(
    "/home/onyxia/work/ddpm/mdlm_hf", trust_remote_code=True).cuda().eval()

B, L, T = 1, 128, 128
mask_id = model.config.vocab_size - 1

In [ ]:
x = mdlm_sample(model, B, L, T, mask_id)
print((x == mask_id).sum().item())   # doit valoir 0
print(tok.decode(x[0]))

# Fine tune DDPM model on a single class

In [ ]:
from torch.utils.data import Subset

CLASS_ID = 3
idx = [i for i, y in enumerate(training_data.targets) if y == CLASS_ID]
data_train_class = Subset(training_data, idx)
print(len(idx))   # 5000

In [ ]:
timesteps = 1000
beta1, beta2 = 1e-4, 0.02

n_feat = 128
height = 32
in_channels = 3

batch_size = 128
lr = 2e-5
epochs = 20

RESUME = False
USE_EMA = True
USE_AMP = True        # fp16, la T4 a des tensor cores fp16 (surtout PAS bf16 :
                      # Turing ne le gère pas nativement, ce serait plus lent)

torch.backends.cudnn.benchmark = True   # shapes fixes -> autotune cuDNN

In [ ]:
dl_class = DataLoader(data_train_class, batch_size=batch_size, shuffle=True,
                      pin_memory=(device.type == "cuda"),
                      num_workers=4, drop_last=True)

model = UNet(in_channels, n_feat).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()
ema = EMA(model, decay=0.99) if USE_EMA else None
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)